In [0]:
try:
    landing_table = dbutils.widgets.get("landing_table")
    history_table = dbutils.widgets.get("history_table")
except Exception as e:
    print(f"Error initializing variables: {e}")
    raise

In [0]:
try:
    spark.sql(f"""
        MERGE INTO {history_table} tgt
        USING (
            SELECT *
            FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY _load_timestamp) AS rn
                FROM {landing_table}
            ) sub
            WHERE rn = 1
        ) src
        ON tgt.id = src.id
        AND tgt._modified_ts < src._load_timestamp
        AND tgt._active_flag = 1

        WHEN MATCHED THEN
            UPDATE SET
                tgt._active_flag = 0,
                tgt._modified_ts = src._load_timestamp
    """)
except Exception as e:
    print(f"Error updating {history_table}: {e}")
    raise

In [0]:
try:
    spark.sql(f"""
        MERGE INTO {history_table} tgt
        USING (
            SELECT *
            FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY _load_timestamp) AS rn
                FROM {landing_table}
            ) sub
            WHERE rn = 1
        ) src
        ON tgt.id = src.id
        AND tgt._created_ts = src._load_timestamp

        WHEN NOT MATCHED
        THEN
            INSERT (
                _id,
                create_date,
                create_mode,
                patient_control_no,
                patient_ctl_no,
                patient,
                patient_name,
                member_id,
                claim_charge,
                claim_payment,
                statement_start,
                statement_end,
                payer_name,
                pay_date,
                payee_name,
                check_number,
                payer_control_no,
                payer_icn,
                payee_npi,
                trans_id,
                id,
                _file_name,
                _created_ts,
                _modified_ts,
                _active_flag
            )
            VALUES (
                src._id,
                src.create_date,
                src.create_mode,
                src.patient_control_no,
                src.patient_ctl_no,
                src.patient,
                src.patient_name,
                src.member_id,
                src.claim_charge,
                src.claim_payment,
                src.statement_start,
                src.statement_end,
                src.payer_name,
                src.pay_date,
                src.payee_name,
                src.check_number,
                src.payer_control_no,
                src.payer_icn,
                src.payee_npi,
                src.trans_id,
                src.id,
                src._file_name,
                src._load_timestamp,
                src._load_timestamp,
                1
            );
    """)
except Exception as e:
    print(f"Error inserting into {history_table}: {e}")
    raise